## Template Method

---

> **In one line.** A template method $T$ is a *fixed composition* of steps living in the base class: some steps are concrete and shared ($s_i$), others are hooks left blank ($\hat{s}_i = \bot$) until a subclass fills them in. The order of $T$ never changes — only the *content* of the hooks varies.

### 1. Steps, hooks, and the bottom symbol

Think of an algorithm not as one monolithic function but as a **sequence of named steps**. Some of those steps are decided once and for all; others are deliberately left as holes. We distinguish them with notation.

A **concrete step** $s_i$ is implemented directly in the base class and shared by every subclass — it is always the same, so it carries no hat. A **hook** $\hat{s}_i$ is an *abstract* step: a placeholder written into the skeleton but left undefined, marked with a hat precisely to signal "this slot is variable." Until a subclass supplies code, a hook equals **bottom**:

$$\hat{s}_i = \bot \qquad \text{(undefined — the base class is incomplete here).}$$

Every subclass repairs this by choosing, for each abstract step, one implementation drawn from the **set of all possible hook fillings** $\mathcal{H}$:

$$\hat{s}_i \in \mathcal{H} \qquad \text{(supplied by the subclass).}$$

So a subclass is nothing more than a choice of values for the bottoms: it turns each $\bot$ into a concrete element of $\mathcal{H}$, leaving the $s_i$ entirely alone.

### 2. The skeleton as a fixed composition

The **template method** $T$ is the algorithm skeleton defined in the base class. It is built by **composition** $\circ$ — applying the steps in a fixed sequence, step $1$ first, then step $2$, and so on. Mixing concrete steps and hooks, the skeleton is:

$$\boxed{\,T(x) \;=\; s_1(x) \circ \hat{s}_2(x) \circ s_3(x) \circ \cdots \circ \hat{s}_n(x)\,}$$

The crucial point is *which letters carry hats and which do not*. The hatted positions are exactly the variation points; the bare positions are frozen machinery shared across all subclasses. Reading the composition left to right gives the data-flow chain through the algorithm:

$$x \;\xrightarrow{\,s_1\,}\; \cdot \;\xrightarrow{\,\hat{s}_2\,}\; \cdot \;\xrightarrow{\,s_3\,}\; \cdots \;\xrightarrow{\,\hat{s}_n\,}\; T(x),$$

where a subclass may repaint the $\hat{s}$ arrows but can never reorder, delete, or insert arrows. Two subclasses therefore differ *only* in the hatted arrows:

$$\hat{s}_i \in \mathcal{H} \quad \text{(filled by subclass)} \qquad \hat{s}_i = \bot \text{ until then.}$$

### 3. Key conditions

1. **Order invariant.** The sequence of steps inside $T$ is fixed by the base class and cannot be reordered by subclasses. The arrangement $s_1 \circ \hat{s}_2 \circ s_3 \circ \cdots \circ \hat{s}_n$ is immutable; only the *content* of each $\hat{s}_i$ is allowed to vary.

2. **Concrete vs. hook.** Concrete steps $s_i$ are shared and implemented exactly once in the base class. Hooks $\hat{s}_i$ are abstract — every subclass **must** supply its own $\hat{s}_i \in \mathcal{H}$, replacing the $\bot$.

3. **Hollywood principle.** *"Don't call us, we'll call you."* Control flows downward only: the base class invokes the hooks from within $T$, never the reverse. A subclass never calls $T$ on itself as part of defining a step — it merely provides the $\hat{s}_i$ that $T$ will reach for.

&nbsp;

> 🍳 A recipe template $T$. Every recipe runs the same fixed order: gather $\to$ prepare $\to$ cook $\to$ plate $\to$ serve. The order is locked. What you actually cook — the hook $\hat{s}_{\text{cook}} \in \mathcal{H}$ — is filled in differently for pasta, soup, or steak, while gathering and serving stay the same shared $s_i$.

### Exercise 09 — Report Generator

---

**Scenario:** All reports follow: gather data ($s_1$) → format header ($\hat{s}_2$) → format body ($\hat{s}_3$) → format footer ($\hat{s}_4$) → output ($s_5$). Headers and bodies differ for CSV vs HTML.

**Your task:** Define `ReportTemplate` with `generate()` as $T$, and two subclasses filling in the hooks $\hat{s}_2, \hat{s}_3, \hat{s}_4$.

```python
class ReportTemplate:
    def generate(self):        # T — fixed order, never overridden
        self.gather_data()     # s_1: concrete
        self.format_header()   # ŝ_2: hook — subclass fills in
        self.format_body()     # ŝ_3: hook
        self.format_footer()   # ŝ_4: hook
        self.output()          # s_5: concrete
```

**Hints**

- Mark hooks with `@abstractmethod` — this formally declares $\hat{s}_i = \bot$ until a subclass fills them in.
- `generate()` must never be overridden by subclasses — it is $T$, the fixed skeleton. Subclasses only touch $\hat{s}_i$.

In [ ]:
from abc import ABC, abstractmethod

# --------------------------------
# Base class: defines T (fixed skeleton). Concrete steps s_i implemented once.

class ReportTemplate(ABC):
    def generate(self):                  # T — fixed order, never overridden
        self.gather_data()               # s_1: concrete
        self.format_header()             # ŝ_2: hook
        self.format_body()               # ŝ_3: hook
        self.format_footer()             # ŝ_4: hook
        self.output()                    # s_5: concrete

    def gather_data(self):               # s_1: shared concrete step
        self._rows = [("Alice", 30), ("Bob", 25)]
        print("gather_data: loaded", len(self._rows), "rows")

    def output(self):                    # s_5: shared concrete step
        print("--- OUTPUT ---")
        print("\n".join(self._lines))

    @abstractmethod
    def format_header(self): ...         # ŝ_2 = ⊥ until subclass fills in
    @abstractmethod
    def format_body(self): ...           # ŝ_3 = ⊥
    @abstractmethod
    def format_footer(self): ...         # ŝ_4 = ⊥

# --------------------------------
# Subclass: ŝ_i ∈ H for CSV

class CSVReport(ReportTemplate):
    def format_header(self):             # ŝ_2 for CSV
        ...
    def format_body(self):               # ŝ_3 for CSV
        ...
    def format_footer(self):             # ŝ_4 for CSV
        ...

# --------------------------------
# Subclass: ŝ_i ∈ H for HTML

class HTMLReport(ReportTemplate):
    def format_header(self):             # ŝ_2 for HTML
        ...
    def format_body(self):               # ŝ_3 for HTML
        ...
    def format_footer(self):             # ŝ_4 for HTML
        ...

# --------------------------------
CSVReport().generate()
print("========")
HTMLReport().generate()

### Exercise 10 — Data Mining Pipeline

---

**Scenario:** $T$ = open ($\hat{s}_1$) → extract ($\hat{s}_2$) → parse ($s_3$) → analyse ($s_4$) → report ($s_5$) → close ($\hat{s}_6$). Opening and closing differ for CSV vs databases; parsing and analysis are shared concrete steps.

**Your task:** Build `DataMiner` template and two concrete subclasses for CSV and Database sources.

```python
miner = CSVDataMiner("data.csv")
miner.mine()    # T — open, extract, parse, analyse, report, close in fixed order
```

**Hints**

- Identify which steps are $s_i$ (shared/concrete) vs $\hat{s}_i$ (hooks). The split: `open()`, `extract()`, `close()` are hooks; `parse()`, `analyse()`, `report()` are concrete.
- `mine()` is $T$ — fix the order in the base class and never override it. Subclasses only fill $\hat{s}_1, \hat{s}_2, \hat{s}_6$.

In [ ]:
from abc import ABC, abstractmethod

# --------------------------------
# Base class: T = ŝ_1 ∘ ŝ_2 ∘ s_3 ∘ s_4 ∘ s_5 ∘ ŝ_6

class DataMiner(ABC):
    def mine(self):                      # T — fixed order, never overridden
        raw = self.open()                # ŝ_1: hook
        data = self.extract(raw)         # ŝ_2: hook
        parsed = self.parse(data)        # s_3: concrete
        result = self.analyse(parsed)    # s_4: concrete
        self.report(result)             # s_5: concrete
        self.close()                     # ŝ_6: hook

    def parse(self, data):               # s_3: shared concrete step
        return [int(x) for x in data]

    def analyse(self, parsed):           # s_4: shared concrete step
        return sum(parsed)

    def report(self, result):            # s_5: shared concrete step
        print("report: total =", result)

    @abstractmethod
    def open(self): ...                  # ŝ_1 = ⊥
    @abstractmethod
    def extract(self, raw): ...          # ŝ_2 = ⊥
    @abstractmethod
    def close(self): ...                 # ŝ_6 = ⊥

# --------------------------------
# Subclass for CSV: ŝ_1, ŝ_2, ŝ_6 ∈ H

class CSVDataMiner(DataMiner):
    def __init__(self, path):
        self._path = path

    def open(self):                      # ŝ_1 for CSV
        ...
    def extract(self, raw):              # ŝ_2 for CSV
        ...
    def close(self):                     # ŝ_6 for CSV
        ...

# --------------------------------
# Subclass for Database: ŝ_1, ŝ_2, ŝ_6 ∈ H

class DatabaseDataMiner(DataMiner):
    def __init__(self, dsn):
        self._dsn = dsn

    def open(self):                      # ŝ_1 for Database
        ...
    def extract(self, raw):              # ŝ_2 for Database
        ...
    def close(self):                     # ŝ_6 for Database
        ...

# --------------------------------
CSVDataMiner("data.csv").mine()
print("========")
DatabaseDataMiner("postgres://localhost/db").mine()